# 🎯 ArcFace Fine-Tuning for Recognition (End-to-End Notebook)

## What you already have
You have a trained checkpoint at:

- `./weights/model_for_inference.pth`

Inside this checkpoint:
- `model_state_dict` → the model weights
- `brand_to_idx`, `idx_to_brand` → label mapping
- `config`, `test_accuracy` → metadata

## What we want to improve
Your current model is a standard classifier:

- `backbone` (ResNet-like)
- `embedding`: **2048 → 512**
- `classifier`: **512 → 2984**

We will **replace `classifier` with ArcFace** (ArcMargin head) and **fine-tune**.

> ⚠️ ArcFace is a training-time improvement.  
> It will **not** improve performance unless you **train/fine-tune** after adding it.


In [1]:
# Imports
import os
import re
import math
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

import torchvision
import torchvision.models as models


# 🧪 Environment Check

## Why this cell exists
We confirm:
- PyTorch + torchvision versions
- Whether CUDA is available
- Which device we will train on (CPU/GPU)

This helps avoid silent performance problems (training on CPU by accident).


In [2]:
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("cuda available:", torch.cuda.is_available())

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


torch: 2.7.1+cu118
torchvision: 0.22.1+cu118
cuda available: True
Using device: cuda


# 📁 Verify Paths & Working Directory

## Why this cell exists
Your checkpoint path uses a relative path:

- `./weights/model_for_inference.pth`

So your notebook **must be running inside** the `recognition/` folder.

## Expected folder layout
```text
recognition/
  weights/
    model_for_inference.pth
  resnet_train.ipynb
  ...


In [3]:

### ✅ Cell 3 — Code
print("CWD:", os.getcwd())

ckpt_path = "./weights/model_for_inference.pth"
print("Checkpoint exists:", os.path.exists(ckpt_path))

print("weights folder files:", os.listdir("./weights"))


CWD: c:\Users\Admin\Documents\GitHub\Thesis-Computer-Vision\recognition
Checkpoint exists: True
weights folder files: ['model_for_inference.pth', 'README.md']


# 📦 Load Checkpoint and Extract the Real Weights

## Why this cell exists
Your `.pth` is not a raw `state_dict`.  
It is a dictionary checkpoint with keys like:

- `model_state_dict`
- `config`
- `brand_to_idx`, `idx_to_brand`

So we must:
1. Load the checkpoint dict
2. Extract: `sd = ckpt["model_state_dict"]`
3. Extract label mapping
4. Compute the number of classes


In [4]:
ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)

print("Checkpoint keys:", list(ckpt.keys()))

sd = ckpt["model_state_dict"]
brand_to_idx = ckpt.get("brand_to_idx", {})
idx_to_brand = ckpt.get("idx_to_brand", {})

num_classes = len(brand_to_idx) if isinstance(brand_to_idx, dict) else 2984

print("num_classes:", num_classes)
print("num tensors in model_state_dict:", len(sd))


Checkpoint keys: ['model_state_dict', 'config', 'test_accuracy', 'brand_to_idx', 'idx_to_brand']
num_classes: 2984
num tensors in model_state_dict: 327


# ✅ Confirm the Model Shapes (Critical Sanity Check)

## Why this cell exists
We confirm the two most important shapes:

- `embedding.weight` should be `(512, 2048)`  
  meaning: `Linear(2048 → 512)`

- `classifier.weight` should be `(2984, 512)`  
  meaning: `Linear(512 → 2984)`

If these match, then our ArcFace model definition will match your checkpoint.


In [5]:
print("embedding.weight shape:", tuple(sd["embedding.weight"].shape))       # expected (512, 2048)
print("classifier.weight shape:", tuple(sd["classifier.weight"].shape))     # expected (2984, 512)


embedding.weight shape: (512, 2048)
classifier.weight shape: (2984, 512)


# 🧱 Auto-Detect the ResNet Backbone Depth

## Why this cell exists
Your checkpoint stores backbone weights like:
- `backbone.layer1.*`
- `backbone.layer2.*`
- `backbone.layer3.*`
- `backbone.layer4.*`

This pattern belongs to the ResNet bottleneck family:
- ResNet50  → `[3, 4, 6, 3]`
- ResNet101 → `[3, 4, 23, 3]`
- ResNet152 → `[3, 8, 36, 3]`

We will detect which one you used by counting block indices in `state_dict`.


In [6]:
def infer_resnet_depth(sd):
    layer_max = {1: -1, 2: -1, 3: -1, 4: -1}
    pat = re.compile(r"^backbone\.layer([1-4])\.(\d+)\.")

    for k in sd.keys():
        m = pat.match(k)
        if m:
            layer = int(m.group(1))
            idx = int(m.group(2))
            layer_max[layer] = max(layer_max[layer], idx)

    blocks = [layer_max[i] + 1 for i in [1, 2, 3, 4]]
    print("Detected blocks per layer:", blocks)

    if blocks == [3, 4, 6, 3]:
        return 50
    if blocks == [3, 4, 23, 3]:
        return 101
    if blocks == [3, 8, 36, 3]:
        return 152
    return 50

resnet_depth = infer_resnet_depth(sd)
print("Detected backbone: ResNet", resnet_depth)


Detected blocks per layer: [3, 4, 6, 3]
Detected backbone: ResNet 50


# 🎯 Define ArcFace (ArcMargin) Head

## What this head does
ArcFace modifies the logit of the **correct class** by adding an angular margin:

- Normalized embedding `e`
- Normalized class weights `W`
- Cosine: `cos(θ)`
- For correct class: `cos(θ + m)`
- Multiply by scale `s`

## Why it helps
ArcFace makes embeddings:
- more compact within the same class
- more separated between different classes

## Common hyperparameters
- `m` (margin): `0.35` to `0.50`
- `s` (scale): `32` to `64`


In [7]:
class ArcMarginProduct(nn.Module):
    def __init__(self, in_features, out_features, s=64.0, m=0.4):
        super().__init__()
        self.s = s
        self.m = m

        self.weight = nn.Parameter(torch.randn(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)

        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, emb, labels=None):
        emb = F.normalize(emb, dim=1)
        W = F.normalize(self.weight, dim=1)

        cosine = F.linear(emb, W).clamp(-1.0, 1.0)

        # Inference: no margin
        if labels is None:
            return cosine * self.s

        sine = torch.sqrt(1.0 - cosine**2)
        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)

        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1), 1.0)

        logits = one_hot * phi + (1.0 - one_hot) * cosine
        return logits * self.s


# 🏗️ Build the New Model (Backbone + Embedding + ArcFace)

## Why this cell exists
We must define a new model that:
- Keeps the same **module names** as your checkpoint (`backbone`, `embedding`)
- Has the same embedding shape (`2048 → 512`)
- Replaces `classifier` with `arcface`

## Architecture
```text
Image
  ↓
ResNet backbone (outputs 2048-d)
  ↓
Embedding: Linear(2048 → 512)
  ↓
ArcFace head: ArcMargin(512 → num_classes)


In [8]:

### ✅ Cell 8 — Code
def build_backbone(depth):
    if depth == 50:
        net = models.resnet50(weights=None)
    elif depth == 101:
        net = models.resnet101(weights=None)
    elif depth == 152:
        net = models.resnet152(weights=None)
    else:
        net = models.resnet50(weights=None)

    net.fc = nn.Identity()
    return net


class RecognitionArcFace(nn.Module):
    def __init__(self, num_classes, emb_dim=512, s=64.0, m=0.4, depth=50):
        super().__init__()
        self.backbone = build_backbone(depth)
        self.embedding = nn.Linear(2048, emb_dim)
        self.arcface = ArcMarginProduct(emb_dim, num_classes, s=s, m=m)

    def forward(self, x, labels=None):
        feat = self.backbone(x)     # [B, 2048]
        emb = self.embedding(feat)  # [B, 512]
        logits = self.arcface(emb, labels)
        return logits, emb


# 🔁 Load Old Weights into the New Model

## Why `strict=False`?
Your old checkpoint contains:
- `classifier.weight`
- `classifier.bias`

But the new model contains:
- `arcface.weight` (new)

So:
- `classifier.*` will show up in `unexpected`
- `arcface.weight` will show up in `missing`

That is correct and expected.


In [9]:
model = RecognitionArcFace(
    num_classes=num_classes,
    emb_dim=512,
    s=64.0,
    m=0.4,
    depth=resnet_depth
)

missing, unexpected = model.load_state_dict(sd, strict=False)

print("Missing keys (expected arcface):", missing)
print("Unexpected keys (expected old classifier):", unexpected)

model = model.to(device)


Missing keys (expected arcface): ['arcface.weight']
Unexpected keys (expected old classifier): ['bn.weight', 'bn.bias', 'bn.running_mean', 'bn.running_var', 'bn.num_batches_tracked', 'classifier.weight', 'classifier.bias']


# ⚙️ Training Setup (Loss + Optimizer)

## Why this cell exists
ArcFace outputs logits, so we can still use **CrossEntropyLoss**.

We set different learning rates:
- Backbone: very small LR (we don't want to destroy pretrained features)
- Embedding: moderate LR
- ArcFace head: larger LR (new parameters need to learn fast)


In [10]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW([
    {"params": model.backbone.parameters(), "lr": 1e-5},
    {"params": model.embedding.parameters(), "lr": 1e-4},
    {"params": model.arcface.parameters(), "lr": 1e-3},
], weight_decay=1e-4)

print("Optimizer ready.")


Optimizer ready.


# 🔥 Fine-Tuning Loop (ArcFace Requires Labels!)

## Key rule
During training you must call:
```python
logits, emb = model(images, labels)


In [11]:

### ✅ Cell 11 — Code
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        logits, emb = model(images, labels)  # labels are required here
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / max(total, 1), correct / max(total, 1)


# Start small first to confirm everything runs
epochs = 3
for epoch in range(1, epochs + 1):
    loss, acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    print(f"Epoch {epoch}/{epochs} | loss={loss:.4f} | acc={acc:.4f}")


NameError: name 'train_loader' is not defined

# 💾 Save the New ArcFace Checkpoint

## Why this cell exists
We保存:
- new model weights
- the original label mapping

So your inference pipeline can still map predictions ↔ brand names.


In [ ]:
save_path = "./weights/model_arcface.pth"

torch.save({
    "model_state_dict": model.state_dict(),
    "brand_to_idx": brand_to_idx,
    "idx_to_brand": idx_to_brand,
    "config": ckpt.get("config", {}),
}, save_path)

print("Saved:", save_path)


# 🔍 Inference Example (No Labels)

## Key rule
During inference, call:
```python
logits, emb = model(images, labels=None)


In [ ]:

### ✅ Cell 13 — Code
model.eval()
with torch.no_grad():
    images, _ = next(iter(train_loader))
    images = images.to(device)

    logits, emb = model(images, labels=None)
    preds = logits.argmax(dim=1)

print("logits shape:", logits.shape)
print("emb shape:", emb.shape)
print("preds shape:", preds.shape)
